# Phase 0 — Zero-shot sertanejo eval

Sanity-check that our v0 base candidates already produce something *sertanejo-shaped* before we invest in fine-tuning.

Runs 20 prompts (5 per sub-genre: raiz / universitário / sofrência / feminejo) through:

1. **`TucanoBR/Tucano-630m`** — v0 default base
2. **`TucanoBR/Tucano-1b1-Instruct`** — v1 stretch target
3. **`rsmonteiro/gpt2-small-portuguese-lyrics`** — prior-art baseline

Designed for **free Colab T4 (16GB)**. Total runtime ≈ 10-15 min.

**At the end:** the three result JSONs are written under `docs/`. Commit them back to the branch and use them to fill `docs/00-zero-shot-samples.md` with your qualitative verdict.


## 1. Setup


In [ ]:
# Confirm GPU is available
!nvidia-smi | head -n 20


In [ ]:
%pip install -q transformers accelerate sentencepiece


In [ ]:
# Clone this repo's branch into Colab
import os, subprocess
REPO = 'https://github.com/guitorte/som.git'
BRANCH = 'claude/ptbr-lyrics-llm-setup-E04UU'
if not os.path.exists('som'):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO], check=True)
%cd som/lyrics-llm
!ls


## 2. Run zero-shot generation

Each cell loads one model, generates for all 20 prompts, writes JSON, then frees VRAM before the next.


### 2a. Tucano-630m (v0 base)


In [ ]:
!python scripts/zero_shot.py \
    --model TucanoBR/Tucano-630m \
    --prompts evaluation/prompts/sertanejo_prompts.json \
    --out docs/00-zero-shot-Tucano-630m.json


### 2b. Tucano-1b1-Instruct (v1 stretch)


In [ ]:
!python scripts/zero_shot.py \
    --model TucanoBR/Tucano-1b1-Instruct \
    --prompts evaluation/prompts/sertanejo_prompts.json \
    --out docs/00-zero-shot-Tucano-1b1-Instruct.json


### 2c. rsmonteiro/gpt2-small-portuguese-lyrics (prior-art baseline)


In [ ]:
!python scripts/zero_shot.py \
    --model rsmonteiro/gpt2-small-portuguese-lyrics \
    --prompts evaluation/prompts/sertanejo_prompts.json \
    --out docs/00-zero-shot-gpt2-small-portuguese-lyrics.json


## 3. Inspect outputs

Quick side-by-side comparison printed per prompt.


In [ ]:
import json, pathlib, textwrap
files = [
    'docs/00-zero-shot-Tucano-630m.json',
    'docs/00-zero-shot-Tucano-1b1-Instruct.json',
    'docs/00-zero-shot-gpt2-small-portuguese-lyrics.json',
]
results = {pathlib.Path(f).stem: json.load(open(f, encoding='utf-8')) for f in files}
for i in range(20):
    sample_ids = {k: v['samples'][i] for k, v in results.items()}
    first = next(iter(sample_ids.values()))
    print('=' * 80)
    print(f"{first['prompt_id']} [{first['sub_genre']}]")
    print('PROMPT:', first['prompt'].strip())
    for tag, sample in sample_ids.items():
        print(f"\n-- {tag} --")
        print(textwrap.indent(sample['completion'].strip(), '   '))
    print()


## 4. Commit results back to the branch

The three JSON files under `docs/` are the artifacts we want to keep.
Authenticate with a Personal Access Token (Settings → Developer settings → Tokens → fine-grained, repo access to `guitorte/som`, write permission).


In [ ]:
# Uncomment, fill in, run.
# !git config user.email 'you@example.com'
# !git config user.name 'Your Name'
# !git add docs/00-zero-shot-*.json
# !git commit -m 'Phase 0: zero-shot sertanejo samples for Tucano-630m, 1b1-Instruct, gpt2-pt-lyrics'
# !git push https://<USERNAME>:<TOKEN>@github.com/guitorte/som.git HEAD:claude/ptbr-lyrics-llm-setup-E04UU


## 5. Verdict

After reading the outputs, edit `docs/00-zero-shot-samples.md` (already templated in the repo) with:

- Which model produced the most sertanejo-shaped output zero-shot.
- Whether Tucano-630m is good enough for v0, or we should escalate to 1b1-Instruct (or step down to 160m if 630m is too unwieldy on Colab T4).
- Any prompt-style adjustments to make before Phase 1.
